In [1]:
import sys
sys.path.insert(1, '../../../scripts/')
from preprocess import preprocess
from preprocess import correct_inputs 

from utils import functions as func
from tqdm import tqdm
from utils import parameters as params
import copy
from utils import metabolites as metab
import pickle
import sympy

import numpy as np
import pandas as pd
import pickle

lp_path = '/data2/hratch/human_me/test_lp/'

ERROR:cobra.io.sbml:No objective coefficients in model. Unclear what should be optimized


In [2]:
jabba = True

counter = 3
base = 2

mu_val = 1e-9

In [4]:
from expression import build_me_model
tme, builder = build_me_model.build_me(minimal_proteome = True, compress_mrna = False, 
                                                unmodeled_protein_frac = None,
                                                model_id = 'toy_me_model')

if jabba:
    for r in tme.reactions:
        if ('EX_' in r.id and r.compartments == {'b'} and r.bounds == (float('-inf'), float('inf'))):
            r._lower_bound = -1000
            r._upper_bound = 1000
    
    
# with open(lp_path + 'working_version_' + str(counter) + '.pickle', 'rb') as handle:
#     tme = pickle.load(handle)

# with open(lp_path + 'notworking_version.pickle', 'rb') as handle:
#     tme = pickle.load(handle)

In [5]:
sln, stat, _ = tme.solve_lp(mu_val = mu_val)

fn = '/data2/hratch/human_me/test_lp/S_matrix.h5'
if stat == 0:
    S = tme.create_stoichiometric_matrix(mu_val = 1, inplace = False, array_type = 'pandas')
    S.to_hdf(fn, key = str(counter), mode = 'a')
    print('Last saved file: {}'.format(counter))
    
    res = pd.DataFrame(data = {'reaction_fluxes': sln[:len(tme.reactions)]})
    res.index = [r.id for r in tme.reactions]
else:
    infeasible_reactions = tme.infeasible_reactions(mu_val = mu_val, sln = sln, stat = stat)
    res = None
    raise ValueError('Model did not solve')
res.loc[[i for i in res.index if 'biomass' in i],:]

Getting MINOS parameters...
Done in 163.498 seconds with status 0


/home/hratch/Projects/human_me/me_env/lib/python3.6/site-packages/tables/path.py:155 NaturalNameWarning: object name is not a valid Python identifier: '2'; it does not match the pattern ``^[a-zA-Z_][a-zA-Z0-9_]*$``; you will not be able to use natural naming to access this object; using ``getattr()`` will still work, though


Last saved file: 2


,reaction_fluxes
biomass_dilution,1.000000e-09
DNA_biomass_to_biomass,1.400000e-11
carbohydrate_biomass_to_biomass,7.100000e-11
lipid_biomass_to_biomass,9.700000e-11
tRNA_biomass_to_biomass,1.132826e-22
rRNA_biomass_to_biomass,2.306195e-14
mRNA_biomass_to_biomass,3.703759e-10
premRNA_biomass_to_biomass,-2.047182e-21
other_rna_biomass_to_biomass,1.703574e-21
DNA_biomass_formation,1.000000e-09


In [6]:
def save_me_model(me_model, counter):
    print('Success, please update git')
    lp_path = '/data2/hratch/human_me/test_lp/'
    with open(lp_path + 'working_version_' + str(counter) + '.pickle', 'wb') as handle:
        pickle.dump(me_model, handle)

def get_changes(S_1, S_0):
    mismatch = np.argwhere(np.not_equal(S_0.values, S_1.values))
    am = S_1.index.tolist()
    mm = {m: {'id': am[m]} for m in sorted(set([t[0] for t in mismatch]))}
    for m in mm:
        mm[m]['reactions'] = sorted(set([t[1] for t in mismatch if t[0] == m]))

    am, rm = S_1.index.tolist(), S_1.columns.tolist()
    mm_2 = {m: {'id': am[m]} for m in sorted(set([t[0] for t in mismatch]))}
    for m in mm_2:
        mm_2[m]['reactions'] = sorted(set(['_'.join(rm[t[1]].split('_')[1:]) if 'HGNC' in rm[t[1]] else rm[t[1]] for t in mismatch if t[0] == m]))
    
    return mismatch, mm, mm_2

In [7]:
S_1 = pd.read_hdf(fn, key = str(counter))
S_0 = pd.read_hdf(fn, key = str(base))

if not S_0.equals(S_1):
    indeces = True
    if indeces:
        if S_1.shape != S_0.shape:
            print('Dimensions are not the same')
            indeces = False
        if len(set(S_1.columns).difference(S_0.columns)) > 0:
            print('Columns are not the same')
            indeces = False
        if len(set(S_1.index).difference(S_0.index)) > 0:
            print('Rows are not the same')
            indeces = False
    if indeces:
        S_1 = S_1.loc[S_0.index, S_0.columns]
        if not S_0.equals(S_1):
            mismatch, mm, mm_2 = get_changes(S_1, S_0)
            print('Dataframes are not equal due to stoichiometric values mismatch, will not save model')
        else:
            save_me_model(tme, counter)
    else:
        print('Dataframes are not equal due to column/row label mismatch, will not save model')
else:
    save_me_model(tme, counter)

Rows are not the same
Dataframes are not equal due to column/row label mismatch, will not save model


In [9]:
set(S_1.index).difference(S_0.index)

{'290067835_complex_n'}

In [8]:
mm.keys()

NameError: name 'mm' is not defined

In [ ]:
m_idx = 1960
mm_2[m_idx]

In [ ]:
mm[m_idx]

In [ ]:
S_1.iloc[m_idx, mm[m_idx]['reactions']]

In [ ]:
S_0.iloc[m_idx, mm[m_idx]['reactions']]

In [ ]:
test = S_1.iloc[m_idx, mm[m_idx]['reactions']]  - S_0.iloc[m_idx, mm[m_idx]['reactions']]
test[abs(test) > 1e-8]

# testing ubiquitin cleavage

In [ ]:
def binary_search(tme, bm_min=-17.111457840000003, bm_max=-0.1, accuracy=0.1):
    feasible_mu = [bm_min]
    infeasible_mu = [bm_max]
    def replace_biomass(biomass_val):
        new_reactions = [r.copy() for r in tme.reactions]
        r_ = [r for r in new_reactions if r.id == 'HGNC:12458_UBIQUITIN_CLEAVAGEc'][0]
        r_.add_metabolites({tme.metabolites.get_by_id('biomass_protein'): biomass_val}, 
                         combine = False)
        if len(r_.check_mass_balance()) == 0:
            test_me = func.ME_Model('test')
            test_me.add_reactions(new_reactions)
            print('Begin solve')
            sln0, stat0, _ = test_me.solve_lp(mu_val = 1e-9)
        if stat0.max() == 0:
            feasible_mu.append(biomass_val)
            return True, sln0, stat0
        elif stat0.max() == 1:
            infeasible_mu.append(biomass_val)
            return False, sln0, stat0
        else:
            raise ValueError('Something went wrong')
    
    while (abs(infeasible_mu[-1] - feasible_mu[-1])) > accuracy:
        print('Current infeasible: {}'.format(infeasible_mu[-1]))
        print('Current feasible: {}'.format(feasible_mu[-1]))
        bool_, sln,stat = replace_biomass((infeasible_mu[-1] + feasible_mu[-1]) * 0.5)
        print('--------------')
    
    return sln, stat, feasible_mu, infeasible_mu
    
    
